In [162]:
import pandas as pd
import numpy as np
import plotly.express as px

In [163]:
acc_df = pd.read_csv("/Users/rblc/create/work/elte/polar/data/acc_260608_1742.csv")
ecg_df = pd.read_csv("/Users/rblc/create/work/elte/polar/data/ecg_260608_1742.csv")

In [164]:
ecg_df.shape, acc_df.shape

((77964, 7), (118908, 9))

In [165]:
ecg_df.head(), acc_df.head()

(   packet_id   device_time     host_time   ecg  sample_idx   sample_time  \
 0          0  5.996202e+11  1.780933e+12  4542           0  5.996202e+11   
 1          0  5.996202e+11  1.780933e+12  4876           1  5.996202e+11   
 2          0  5.996202e+11  1.780933e+12  5099           2  5.996202e+11   
 3          0  5.996202e+11  1.780933e+12  5267           3  5.996202e+11   
 4          0  5.996202e+11  1.780933e+12  5511           4  5.996202e+11   
 
      time_ms  
 0   0.000000  
 1   7.692383  
 2  15.384644  
 3  23.076904  
 4  30.769287  ,
    packet_id   device_time     host_time    x  y    z  sample_idx  \
 0          0  5.996202e+11  1.780933e+12 -626  1  732           0   
 1          0  5.996202e+11  1.780933e+12 -629 -3  730           1   
 2          0  5.996202e+11  1.780933e+12 -630  0  738           2   
 3          0  5.996202e+11  1.780933e+12 -632  2  734           3   
 4          0  5.996202e+11  1.780933e+12 -634 -1  727           4   
 
     sample_time 

In [166]:
ecg_df = ecg_df.sort_values("time_ms")
acc_df = acc_df.sort_values("time_ms")

recording = pd.merge_asof(
    ecg_df,
    acc_df[["time_ms", "x", "y", "z"]],
    on="time_ms",
    direction="nearest"
)
recording = recording[["time_ms", "ecg", "x", "y", "z"]]
recording["acc_mag"] = np.sqrt(
    recording["x"]**2 +
    recording["y"]**2 +
    recording["z"]**2)

In [167]:
recording

,time_ms,ecg,x,y,z,acc_mag
0,0.000000,4542,-626,1,732,963.172363
1,7.692383,4876,-630,0,738,970.331902
2,15.384644,5099,-632,2,734,968.598988
3,23.076904,5267,-636,2,724,963.678370
4,30.769287,5511,-638,0,719,961.251788
...,...,...,...,...,...,...
77959,599731.097900,-90,-686,-5,682,967.339134
77960,599738.790283,-79,-680,-6,693,970.919667
77961,599746.482544,-109,-681,-6,687,967.349988
77962,599754.174927,-134,-672,-4,681,956.745003


In [168]:
import neurokit2 as nk

fs = 130

_, info = nk.ecg_peaks(
    recording["ecg"],
    sampling_rate=fs
)

rpeaks = info["ECG_R_Peaks"]

recording["r_peak"] = 0
recording.loc[rpeaks, "r_peak"] = 1

In [169]:
recording

,time_ms,ecg,x,y,z,acc_mag,r_peak
0,0.000000,4542,-626,1,732,963.172363,0
1,7.692383,4876,-630,0,738,970.331902,0
2,15.384644,5099,-632,2,734,968.598988,0
3,23.076904,5267,-636,2,724,963.678370,0
4,30.769287,5511,-638,0,719,961.251788,0
...,...,...,...,...,...,...,...
77959,599731.097900,-90,-686,-5,682,967.339134,0
77960,599738.790283,-79,-680,-6,693,970.919667,0
77961,599746.482544,-109,-681,-6,687,967.349988,0
77962,599754.174927,-134,-672,-4,681,956.745003,0


In [171]:
segment = recording.query(
    "30000 <= time_ms <= 40000"
)

peaks = segment[segment["r_peak"] == 1]

fig = px.line(
    segment,
    x="time_ms",
    y="ecg"
)

fig.add_scatter(
    x=peaks["time_ms"],
    y=peaks["ecg"],
    mode="markers",
    name="R peaks"
)

fig.show()

In [172]:
import numpy as np
import pandas as pd

# R-peak times
rpeak_times = recording.loc[recording["r_peak"] == 1, "time_ms"].to_numpy()

# -----------------------------
# 1) Beat-to-beat HR / HRV
# -----------------------------
rr_ms = np.diff(rpeak_times)

hrv_df = pd.DataFrame({
    "time_ms": rpeak_times[1:],
    "rr_ms": rr_ms,
    "bpm_instant": 60000 / rr_ms
})

# -----------------------------
# 2) 1-second sport-watch-like BPM
# -----------------------------
bpm_df = (
    hrv_df
    .set_index(pd.to_timedelta(hrv_df["time_ms"], unit="ms"))
    .resample("1s")
    .mean(numeric_only=True)
    .interpolate()
    .reset_index(drop=True)
)

bpm_df["time_ms"] = np.arange(len(bpm_df)) * 1000
bpm_df["bpm"] = bpm_df["bpm_instant"].rolling(
    window=5,
    center=True,
    min_periods=1
).mean()

bpm_df = bpm_df[["time_ms", "bpm"]]

In [ ]:
px.line(
    hrv_df,
    x="time_ms",
    y="rr_ms",
    title="RR Intervals Over Time"
)

In [ ]:
px.line(
    bpm_df,
    x="time_ms",
    y="bpm",
    title="Heart Rate (BPM) Over Time"
)

In [ ]:
px.line(
    hrv_df,
    x="time_ms",
    y="rr_ms",
    title="RR Intervals Over Time (HRV View)"
)

In [ ]:
rr = hrv_df["rr_ms"]

px.scatter(
    x=rr[:-1],
    y=rr[1:],
    title="Poincare Plot of RR Intervals",
    labels={"x": "RR(n) [ms]", "y": "RR(n+1) [ms]"}
)

In [ ]:
rr_diff = np.diff(hrv_df["rr_ms"])

hrv_df["rmssd_30"] = (
    pd.Series(rr_diff**2)
    .rolling(30)
    .mean()
    .pow(0.5)
)

px.line(
    hrv_df.iloc[1:],
    x="time_ms",
    y="rmssd_30",
    title="RMSSD Over Time"
)